# 01 — Data Audit

Schema, data dictionary, quality checks and the leakage probe.

The audit runs **before** any modelling. Its purpose is to find the problems that a `df.info()` pass does not: this dataset reports zero nulls while `TotalCharges` has 11 blank strings, and it has zero duplicate ids while carrying feature-identical rows with contradictory labels.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))
warnings.filterwarnings('ignore')

from src.config import load_config
cfg = load_config()

## Raw schema and data dictionary

In [2]:
from src.data.loader import load_raw, clean_raw, deduplicate
from src.data.schema import DATA_DICTIONARY, TARGET
import pandas as pd

raw = load_raw(cfg)
print(raw.shape)
raw.head()

(7043, 21)


,customerID,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,...,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,Yes,No,No,No phone service,DSL,No,Yes,No,...,No,No,Month-to-month,Yes,Electronic check,0,1,29.85,29.85,No
1,5575-GNVDE,Male,No,No,Yes,No,DSL,Yes,No,Yes,...,No,No,One year,No,Mailed check,0,34,56.95,1889.50,No
2,3668-QPYBK,Male,No,No,Yes,No,DSL,Yes,Yes,No,...,No,No,Month-to-month,Yes,Mailed check,0,2,53.85,108.15,Yes
3,7795-CFOCW,Male,No,No,No,No phone service,DSL,Yes,No,Yes,...,No,No,One year,No,Bank transfer (automatic),0,45,42.30,1840.75,No
4,9237-HQITU,Female,No,No,Yes,No,Fiber optic,No,No,No,...,No,No,Month-to-month,Yes,Electronic check,0,2,70.70,151.65,Yes


In [3]:
pd.DataFrame([{'column': s.name, 'kind': s.kind,
               'description': s.description, 'notes': s.notes}
              for s in DATA_DICTIONARY])

,column,kind,description,notes
0,customerID,identifier,Unique customer reference.,Dropped before modelling: an identifier carrie...
1,gender,categorical,Female / Male.,
2,SeniorCitizen,binary,1 if the customer is a senior citizen.,
3,Partner,categorical,Has a partner.,
4,Dependents,categorical,Has dependents.,
5,tenure,numeric,Months the customer has been with the company.,0 for 11 never-billed customers; see the missi...
6,PhoneService,categorical,Subscribes to phone service.,
7,MultipleLines,categorical,Multiple phone lines.,"'No phone service' is structural, set by Phone..."
8,InternetService,categorical,DSL / Fiber optic / No.,
9,OnlineSecurity,categorical,Online-security add-on.,'No internet service' is structural.


## Target definition and balance

`Churn == 'Yes'` is the positive class. Note what the base rate implies: a model predicting 'nobody churns' scores that accuracy while finding no churners at all.

In [4]:
y = (raw[TARGET] == cfg.positive_label).astype(int)
print(f'churn rate           : {y.mean():.4f}')
print(f'majority-class acc.  : {1 - y.mean():.4f}')
print(f'imbalance ratio      : 1:{(1-y.mean())/y.mean():.2f}')

churn rate           : 0.2654
majority-class acc.  : 0.7346
imbalance ratio      : 1:2.77


## The missing values that `isna()` does not show

`TotalCharges` ships as text. Its blanks are whitespace, and they are **not** missing at random.

In [5]:
print('pandas-visible nulls in the raw frame:')
print(raw.isna().sum()[lambda s: s > 0])

blank = raw[raw['TotalCharges'].isna()]
print(f'\nblank TotalCharges: {len(blank)}')
print(f'their tenure values: {sorted(blank["tenure"].unique())}')
print(f'their churn values : {blank[TARGET].unique().tolist()}')
print('\n=> never-billed customers. 0.0 is the true amount billed, '
      'not an imputed guess.')

pandas-visible nulls in the raw frame:
TotalCharges    11
dtype: int64

blank TotalCharges: 11
their tenure values: [np.int64(0)]
their churn values : ['No']

=> never-billed customers. 0.0 is the true amount billed, not an imputed guess.


## Duplicates, label conflicts and contamination

In [6]:
cleaned = clean_raw(raw)
deduped, n_removed, n_conflicting = deduplicate(cleaned)
print(f'exact duplicate rows        : {raw.duplicated().sum()}')
print(f'duplicate customerIDs       : {raw[cfg.id_column].duplicated().sum()}')
print(f'safe duplicates removed     : {n_removed}')
print(f'label-conflicting rows kept : {n_conflicting}')
print(f'=> irreducible noise floor  : {n_conflicting/len(cleaned):.4%} of rows')

exact duplicate rows        : 0
duplicate customerIDs       : 0
safe duplicates removed     : 16
label-conflicting rows kept : 42
=> irreducible noise floor  : 0.5963% of rows


## The full audit, including the executable leakage probe

Every feature is scored **alone** under cross-validation. A feature approaching AUC 1.0 would be the target in disguise. This is the difference between claiming there is no leakage and testing for it.

In [7]:
from src.data import audit
res = audit.run(cfg)

rows x cols                : 7,043 x 21
churn rate                 : 26.54%
missing (incl. whitespace) : {'TotalCharges': 11}
exact dups / dup ids       : 0 / 0
feature dups               : 40  (removed 16)
label conflicts            : 42 rows in 18 groups  -> noise floor 0.5963%
train/test contamination   : 0
invalid ranges/categories  : none / none
structural inconsistencies : none
TotalCharges R^2 vs tenure*Monthly : 0.9991
top single-feature AUC     : Contract=0.7391, tenure=0.7319, OnlineSecurity=0.7055, TechSupport=0.7035, InternetService=0.6953
leakage suspects           : none

[written] /projects/sandbox/aiml-portfolio/aiml3-churn-explainability/reports/data_audit.md


In [8]:
import pandas as pd
pd.Series(res.single_feature_auc, name='cv_roc_auc_alone').head(10)

Contract            0.739094
tenure              0.731923
OnlineSecurity      0.705484
TechSupport         0.703512
InternetService     0.695289
OnlineBackup        0.677707
PaymentMethod       0.671236
DeviceProtection    0.670690
MonthlyCharges      0.661996
TotalCharges        0.648166
Name: cv_roc_auc_alone, dtype: float64

### What cannot be tested

The dataset is a single snapshot with **no event timestamps**. So there is no way to verify empirically that a column was recorded before the churn decision, and no out-of-time split is possible. Post-churn risk is assessed per column from semantics — see `reports/data_audit.md`.